# TS-ResNet50 Training and Import Analysis

This notebook is the repo-runnable home for the main `TS-ResNet50` experiment.

It combines three roles in one place:
- local repo training and evaluation using the shared training scripts
- imported Kaggle artifact inspection and checkpoint remapping
- local score-sweep analysis on top of the imported or locally trained checkpoint

The heavy training run originally happened on Kaggle because `ResNet50` teacher distillation is expensive, but this notebook is still runnable from the repo for anyone who wants to reproduce or extend the experiment locally.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]
REPO_ROOT = None
for candidate in candidate_roots:
    if (candidate / "src" / "wafer_defect").exists() and (candidate / "configs").exists():
        REPO_ROOT = candidate
        break

if REPO_ROOT is None:
    raise RuntimeError("Could not locate repo root containing src/wafer_defect and configs/")

SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from wafer_defect.config import load_toml
from wafer_defect.data.wm811k import WaferMapDataset
from wafer_defect.evaluation.reconstruction_metrics import summarize_threshold_metrics, sweep_threshold_metrics
from wafer_defect.models.ts_distillation import build_ts_distillation_from_config
from wafer_defect.scoring import spatial_max, spatial_mean, topk_spatial_mean

REPO_ROOT

In [ ]:
ARTIFACT_DIR = REPO_ROOT / "artifacts" / "x64" / "ts_resnet50"
CONFIG_PATH = REPO_ROOT / "configs" / "training" / "train_ts_resnet50_kaggle.toml"
CHECKPOINT_PATH = ARTIFACT_DIR / "best_model.pt"
CONVERTED_CHECKPOINT_PATH = ARTIFACT_DIR / "best_model_local_format.pt"
EVALUATION_DIR = ARTIFACT_DIR / "evaluation"
LOCAL_EVALUATION_DIR = ARTIFACT_DIR / "evaluation_local"
THRESHOLD_QUANTILE = 0.95

RUN_LOCAL_TRAINING = False
RUN_DEFAULT_EVALUATION = False
RUN_IMPORT_REMAP = True
RUN_LOCAL_RE_EVALUATION = True
RUN_SCORE_SWEEP = True

SCORE_SWEEP_WEIGHTS = [
    (1.0, 1.0),
    (1.0, 0.0),
    (0.0, 1.0),
    (2.0, 1.0),
    (1.0, 2.0),
    (1.0, 0.5),
    (0.5, 1.0),
]
SCORE_SWEEP_REDUCTIONS = [
    ("mean", None),
    ("max", None),
    ("topk_mean", 0.01),
    ("topk_mean", 0.05),
    ("topk_mean", 0.10),
    ("topk_mean", 0.20),
]

config = load_toml(CONFIG_PATH)
image_size = int(config["data"].get("image_size", 64))
batch_size = int(config["data"].get("batch_size", 64))
num_workers = int(config["data"].get("num_workers", 0))
requested_device = str(config["training"].get("device", "auto"))
resolved_device_name = "cuda" if requested_device == "auto" and torch.cuda.is_available() else requested_device
device = torch.device(resolved_device_name)

print(f"Artifact dir: {ARTIFACT_DIR}")
print(f"Config path: {CONFIG_PATH.name}")
print(f"Resolved device: {device}")
display(config)

In [ ]:
def stream_command(command, cwd):
    print("Running:", " ".join(str(part) for part in command))
    process = subprocess.Popen(
        [str(part) for part in command],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)


if RUN_LOCAL_TRAINING:
    ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    stream_command(
        [
            sys.executable,
            "-u",
            REPO_ROOT / "scripts" / "train_ts_distillation.py",
            "--config",
            CONFIG_PATH,
        ],
        cwd=REPO_ROOT,
    )
else:
    print(f"Skipping local training. Expecting checkpoint at {CHECKPOINT_PATH}")

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT_PATH}")

In [ ]:
imported_summary_path = EVALUATION_DIR / "summary.json"
if not imported_summary_path.exists():
    imported_summary_path = ARTIFACT_DIR / "summary.json"

history_path = ARTIFACT_DIR / "history.json"

if RUN_DEFAULT_EVALUATION:
    EVALUATION_DIR.mkdir(parents=True, exist_ok=True)
    stream_command(
        [
            sys.executable,
            REPO_ROOT / "scripts" / "evaluate_reconstruction_model.py",
            "--checkpoint",
            CHECKPOINT_PATH,
            "--config",
            CONFIG_PATH,
            "--model-type",
            "ts_distillation",
            "--threshold-quantile",
            str(THRESHOLD_QUANTILE),
            "--output-dir",
            EVALUATION_DIR,
        ],
        cwd=REPO_ROOT,
    )

if not imported_summary_path.exists():
    raise FileNotFoundError(f"Summary not found: {imported_summary_path}")
if not history_path.exists():
    raise FileNotFoundError(f"History not found: {history_path}")

history_df = pd.read_json(history_path)
imported_summary = json.loads(imported_summary_path.read_text(encoding="utf-8"))

display(pd.Series(imported_summary.get("train_summary", {})))
display(pd.Series(imported_summary["metrics_at_validation_threshold"]))
display(pd.Series(imported_summary["best_threshold_sweep"]))
display(history_df.tail())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
history_df.plot(x="epoch", y=["train_loss", "val_loss"], ax=axes[0], title="TS-Res50 Loss")
history_df.plot(
    x="epoch",
    y=["train_distillation_loss", "val_distillation_loss", "train_autoencoder_loss", "val_autoencoder_loss"],
    ax=axes[1],
    title="TS-Res50 Component Losses",
)
plt.tight_layout()
plt.show()

In [ ]:
def remap_kaggle_ts_checkpoint(kaggle_checkpoint_path, config_path, image_size):
    config = load_toml(config_path)
    model = build_ts_distillation_from_config(config, image_size=image_size)
    checkpoint = torch.load(kaggle_checkpoint_path, map_location="cpu")
    remapped_state = dict(checkpoint["model_state_dict"])

    if "auto_map_scale" in remapped_state:
        remapped_state["autoencoder_map_scale"] = remapped_state.pop("auto_map_scale")

    layer_aliases = [
        ("teacher.layer1.", "teacher.layers.0."),
        ("teacher.layer2.", "teacher.layers.1."),
        ("teacher.layer3.", "teacher.layers.2."),
        ("teacher.layer4.", "teacher.layers.3."),
    ]
    for old_prefix, new_prefix in layer_aliases:
        for key, value in list(remapped_state.items()):
            if key.startswith(old_prefix):
                remapped_state[new_prefix + key[len(old_prefix):]] = value

    model.load_state_dict(remapped_state, strict=True)
    converted_checkpoint = {
        "epoch": int(checkpoint.get("best_epoch", 0)),
        "model_state_dict": model.state_dict(),
        "config": config,
        "best_epoch": int(checkpoint.get("best_epoch", 0)),
        "best_val_loss": float(checkpoint.get("best_val_loss", 0.0)),
        "history": checkpoint.get("history", []),
        "student_map_scale": float(checkpoint.get("student_map_scale", checkpoint.get("train_summary", {}).get("student_map_scale", 1.0))),
        "autoencoder_map_scale": float(checkpoint.get("auto_map_scale", checkpoint.get("train_summary", {}).get("auto_map_scale", 1.0))),
    }
    return converted_checkpoint, model


if RUN_IMPORT_REMAP or not CONVERTED_CHECKPOINT_PATH.exists():
    converted_checkpoint, converted_model = remap_kaggle_ts_checkpoint(CHECKPOINT_PATH, CONFIG_PATH, image_size=image_size)
    torch.save(converted_checkpoint, CONVERTED_CHECKPOINT_PATH)
    print(f"Saved local-format checkpoint to {CONVERTED_CHECKPOINT_PATH}")
    print(f"best_epoch={converted_checkpoint['best_epoch']}")
    print(f"best_val_loss={converted_checkpoint['best_val_loss']:.6f}")
else:
    converted_checkpoint = torch.load(CONVERTED_CHECKPOINT_PATH, map_location="cpu")
    converted_model = build_ts_distillation_from_config(config, image_size=image_size)
    converted_model.load_state_dict(converted_checkpoint["model_state_dict"], strict=True)
    print(f"Loaded existing local-format checkpoint: {CONVERTED_CHECKPOINT_PATH}")

In [ ]:
metadata_path = REPO_ROOT / config["data"]["metadata_csv"]
metadata = pd.read_csv(metadata_path)
display(metadata.groupby(["split", "is_anomaly"]).size().rename("count").reset_index())

val_dataset = WaferMapDataset(metadata_path, split="val", image_size=image_size)
test_dataset = WaferMapDataset(metadata_path, split="test", image_size=image_size)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
print(f"val={len(val_dataset)}, test={len(test_dataset)}")

def collect_scores_local(model, dataloader, device):
    rows = []
    model.eval()
    with torch.inference_mode():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            scores = model(inputs)
            for score, label in zip(scores.cpu().tolist(), labels.tolist()):
                rows.append({"score": float(score), "is_anomaly": int(label)})
    return pd.DataFrame(rows)

converted_model.to(device)
converted_model.eval()

## Local Re-evaluation and Score Sweep

This section re-evaluates the imported checkpoint on the repo split and then sweeps score-composition settings without retraining.

That lets us separate training quality from score calibration.

In [ ]:
if RUN_LOCAL_RE_EVALUATION:
    LOCAL_EVALUATION_DIR.mkdir(parents=True, exist_ok=True)
    val_scores_df = collect_scores_local(converted_model, val_loader, device)
    test_scores_df = collect_scores_local(converted_model, test_loader, device)
    threshold = float(val_scores_df.loc[val_scores_df["is_anomaly"] == 0, "score"].quantile(THRESHOLD_QUANTILE))
    labels = test_scores_df["is_anomaly"].to_numpy()
    scores = test_scores_df["score"].to_numpy()
    metrics = summarize_threshold_metrics(labels, scores, threshold)
    threshold_sweep_df, best_sweep = sweep_threshold_metrics(labels, scores)

    val_scores_df.to_csv(LOCAL_EVALUATION_DIR / "val_scores.csv", index=False)
    test_scores_df.to_csv(LOCAL_EVALUATION_DIR / "test_scores.csv", index=False)
    threshold_sweep_df.to_csv(LOCAL_EVALUATION_DIR / "threshold_sweep.csv", index=False)

    local_summary = {
        "model_type": "ts_distillation",
        "checkpoint": str(CONVERTED_CHECKPOINT_PATH),
        "threshold_quantile": THRESHOLD_QUANTILE,
        "threshold": threshold,
        "metrics_at_validation_threshold": metrics,
        "best_threshold_sweep": best_sweep,
    }
    (LOCAL_EVALUATION_DIR / "summary.json").write_text(json.dumps(local_summary, indent=2), encoding="utf-8")
else:
    local_summary = json.loads((LOCAL_EVALUATION_DIR / "summary.json").read_text(encoding="utf-8"))

comparison_df = pd.DataFrame([
    {
        "source": "imported_kaggle_summary",
        **imported_summary["metrics_at_validation_threshold"],
        "best_sweep_f1": imported_summary["best_threshold_sweep"]["f1"],
    },
    {
        "source": "local_re_evaluation",
        **local_summary["metrics_at_validation_threshold"],
        "best_sweep_f1": local_summary["best_threshold_sweep"]["f1"],
    },
])
display(comparison_df.round(6))

def collect_normalized_maps(model, dataloader, device):
    student_maps = []
    auto_maps = []
    labels = []
    with torch.inference_mode():
        for inputs, batch_labels in dataloader:
            inputs = inputs.to(device)
            student_map, auto_map = model.raw_anomaly_maps(inputs)
            student_maps.append((student_map / model.student_map_scale.clamp_min(1e-6)).cpu())
            auto_maps.append((auto_map / model.autoencoder_map_scale.clamp_min(1e-6)).cpu())
            labels.append(batch_labels.cpu())
    return torch.cat(student_maps, dim=0), torch.cat(auto_maps, dim=0), torch.cat(labels, dim=0).numpy()

def reduce_anomaly_map(anomaly_map, reduction, topk_ratio):
    if reduction == "mean":
        return spatial_mean(anomaly_map)
    if reduction == "max":
        return spatial_max(anomaly_map)
    return topk_spatial_mean(anomaly_map, topk_ratio=topk_ratio)

if RUN_SCORE_SWEEP:
    val_student_maps, val_auto_maps, val_labels = collect_normalized_maps(converted_model, val_loader, device)
    test_student_maps, test_auto_maps, test_labels = collect_normalized_maps(converted_model, test_loader, device)

    score_sweep_rows = []
    for student_weight, auto_weight in SCORE_SWEEP_WEIGHTS:
        val_map = student_weight * val_student_maps + auto_weight * val_auto_maps
        test_map = student_weight * test_student_maps + auto_weight * test_auto_maps
        for reduction, topk_ratio in SCORE_SWEEP_REDUCTIONS:
            val_scores = reduce_anomaly_map(val_map, reduction, topk_ratio).numpy()
            test_scores = reduce_anomaly_map(test_map, reduction, topk_ratio).numpy()
            threshold = float(pd.Series(val_scores[val_labels == 0]).quantile(THRESHOLD_QUANTILE))
            metrics = summarize_threshold_metrics(test_labels, test_scores, threshold)
            _, best_sweep = sweep_threshold_metrics(test_labels, test_scores)
            score_sweep_rows.append(
                {
                    "name": f"s{student_weight:g}_a{auto_weight:g}_{reduction}" + ("" if topk_ratio is None else f"_r{topk_ratio:.2f}"),
                    "student_weight": student_weight,
                    "auto_weight": auto_weight,
                    "reduction": reduction,
                    "topk_ratio": np.nan if topk_ratio is None else topk_ratio,
                    "threshold": threshold,
                    "precision": metrics["precision"],
                    "recall": metrics["recall"],
                    "f1": metrics["f1"],
                    "auroc": metrics["auroc"],
                    "auprc": metrics["auprc"],
                    "predicted_anomalies": metrics["predicted_anomalies"],
                    "best_sweep_f1": best_sweep["f1"],
                }
            )
    score_sweep_df = pd.DataFrame(score_sweep_rows).sort_values(["f1", "auroc"], ascending=False).reset_index(drop=True)
    score_sweep_df.to_csv(LOCAL_EVALUATION_DIR / "score_sweep_summary.csv", index=False)
    selected_variant = score_sweep_df.iloc[0].to_dict()
    (LOCAL_EVALUATION_DIR / "selected_score_variant.json").write_text(json.dumps(selected_variant, indent=2), encoding="utf-8")
else:
    score_sweep_df = pd.read_csv(LOCAL_EVALUATION_DIR / "score_sweep_summary.csv")
    selected_variant = json.loads((LOCAL_EVALUATION_DIR / "selected_score_variant.json").read_text(encoding="utf-8"))

display(score_sweep_df.head(15).round(6))
display(pd.Series(selected_variant))

In [ ]:
default_row = pd.DataFrame([
    {
        "name": "imported_default_score",
        "precision": imported_summary["metrics_at_validation_threshold"]["precision"],
        "recall": imported_summary["metrics_at_validation_threshold"]["recall"],
        "f1": imported_summary["metrics_at_validation_threshold"]["f1"],
        "auroc": imported_summary["metrics_at_validation_threshold"]["auroc"],
        "auprc": imported_summary["metrics_at_validation_threshold"]["auprc"],
        "best_sweep_f1": imported_summary["best_threshold_sweep"]["f1"],
    }
])
selected_row = pd.DataFrame([
    {
        "name": selected_variant["name"],
        "precision": selected_variant["precision"],
        "recall": selected_variant["recall"],
        "f1": selected_variant["f1"],
        "auroc": selected_variant["auroc"],
        "auprc": selected_variant["auprc"],
        "best_sweep_f1": selected_variant["best_sweep_f1"],
    }
])
summary_comparison_df = pd.concat([default_row, selected_row], ignore_index=True)
display(summary_comparison_df.round(6))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(summary_comparison_df["name"], summary_comparison_df["f1"], color=["#8d99ae", "#2a9d8f"])
axes[0].set_title("Validation-Threshold F1")
axes[0].tick_params(axis="x", rotation=15)
axes[1].bar(summary_comparison_df["name"], summary_comparison_df["auroc"], color=["#8d99ae", "#2a9d8f"])
axes[1].set_title("AUROC")
axes[1].tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()